# 6. Exports

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Khoshkhah/duckOSM/blob/main/docs/notebooks/06_exports.ipynb) <a href="https://khoshkhah.github.io/duckOSM/notebooks/06_exports.ipynb" download><img src="https://img.shields.io/badge/Download-notebook-F37626?logo=jupyter&logoColor=white" alt="Download the notebook"></a>

<img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/banner-6.svg" alt="duckOSM notebook manual, chapter 6: Exports" width="100%">

| **You learn** | which format each group of users needs; the export calls of a `Network`; one mode or all modes; how you check each file; why each format keeps the `edge_id` |
|---|---|
| **Time** | about 3 minutes |
| **Extras** | `duckosm[widget,viz,sim]` (the install cell installs them) |
| **Next** | [7. Routing](07_routing.md): the fastest route between two roads, two points or two names |

In [1]:
# Install duckOSM when it is not installed (for example in Colab). It takes approximately 1 minute.
# Until duckOSM 0.9.0 is on PyPI, the SDK of these notebooks is in the nightly wheel of main (the GitHub pre-release "nightly").
import importlib.util, json, urllib.request
def nightly_wheel():       # the address of the newest nightly wheel
    release = json.load(urllib.request.urlopen("https://api.github.com/repos/Khoshkhah/duckOSM/releases/tags/nightly"))
    return next(a["browser_download_url"] for a in release["assets"] if a["name"].endswith(".whl"))
if importlib.util.find_spec("duckosm") is None:      # not installed: install it with the extras of this notebook
    %pip install -q "duckosm[widget,viz,sim] @ {nightly_wheel()}"
import duckosm             # the SDK
if not hasattr(duckosm, "network"):                    # an old duckOSM (0.8.0 or older), without the SDK
    %pip install -q --force-reinstall --no-deps "duckosm @ {nightly_wheel()}"
    raise SystemExit("duckOSM is updated: restart the session (in Colab: Runtime > Restart session), then run this cell again")
print("duckosm", duckosm.__version__)    # the version

duckosm 0.8.0


## Why a network needs exports

A duckOSM network is a DuckDB file. Many tools cannot read it: a traffic simulator reads its own XML, a planning tool
reads GMNS tables, a GIS reads a GeoPackage. An **export** writes the network in the format of such a tool. Each export
is one call of the `Network`, and each call writes the files and gives a dict of their paths and their numbers.

| Format | Who uses it | The call | Module | How this chapter checks the file | Chapter |
|---|---|---|---|---|---|
| SUMO | traffic engineers who simulate each vehicle (microscopic simulation) | `net.to_sumo` | `duckosm.sim` | SUMO's `netconvert` reads the plain XML and writes the `.net.xml` | 12 |
| MATSim | researchers who simulate the trips of a whole population (agent-based simulation) | `net.to_matsim` | `duckosm.sim` | the XML has one link for each edge | 12 |
| GMNS | transport planners (path4gmns, AequilibraE, DTALite) | `net.to_gmns` | `duckosm.planning` | `duckosm.planning.check_gmns` runs its checks on the tables | 13 |
| OpenDRIVE | driving simulators and HD-map tools (CARLA, esmini) | `net.to_opendrive` | `duckosm.hdmap` | the XML has one road for each edge | 14 |
| GeoPackage, GeoParquet, shapefile | GIS users (QGIS, ArcGIS, GeoPandas) | `net.to_gis` | `duckosm.gis` | DuckDB reads each layer back and counts its rows | 10 |
| QuackOSM table | data engineers who want all OSM features with their tags | `duckosm.gis.quack` | `duckosm.gis` | DuckDB reads the table and counts the features of each OSM type | 10 |
| PMTiles | web maps (MapLibre, OpenLayers) | `duckosm quack --pmtiles` | the command | the file starts with the text `PMTiles` | 10 |

The chapters in the last column use each format in its own field. This chapter writes each format one time and
checks it.

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-key.svg" width="18" height="18" alt=""> **Key idea**
>
> In each format, the id of a road is the duckOSM `edge_id`: the id of a SUMO edge, of a MATSim link, of a GMNS link, of an OpenDRIVE road and of a row of the GIS layer. So the results of each tool (a flow, a speed, a count) join back to the network and to your own data by `edge_id`.

## The parts of an export call

The export calls of a `Network` have the same form:

| Part | Does |
|---|---|
| the first argument | the output: a folder (`to_sumo`) or a file (`to_matsim`, `to_gmns`, `to_opendrive`, `to_gis`) |
| `mode` | the mode to export. `net.driving.to_sumo(...)` is the same as `net.to_sumo(..., mode="driving")` |
| `progress` | the progress line; `False` shows nothing |
| the other keywords | the options of the format, from the function of its module |
| the result | a dict: the paths of the files and the numbers (nodes, edges, links, layers) |

Each call is a function of a module too: `net.to_sumo(...)` is `duckosm.sim.to_sumo(net, ...)`. Some formats are of
one mode, and some take all modes:

| Call | Without `mode` |
|---|---|
| `to_sumo`, `to_opendrive` | the driving network |
| `to_matsim` | the driving network; `mode` can also be a list of modes |
| `to_gmns`, `to_gis` | all the modes of the file |

## Build the network of Monaco

The chapter exports the driving network of Monaco. The summary gives the number of edges and nodes, so you can compare
each export with it.

In [2]:
net = duckosm.network(osm_id=2220322, modes=["driving"], name="monaco")    # get and build the driving network of Monaco
net.summary()                                         # the number of edges, to compare with each export

cut: 34461 nodes, 5685 ways, 339 relations

Output: ~/.cache/duckosm/work/monaco-20261120-194240-0416/monaco.duckdb

,mode,edges,nodes,km,oneway_km
0,driving,2840,1768,96.2,35.5


> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-output.svg" width="18" height="18" alt=""> **What the output shows**
>
> The driving network of Monaco has 2,840 edges and 1,768 nodes, 96.2 km of road. Each export below gives its own count, and the count must agree with these numbers.

## SUMO

[SUMO](https://eclipse.dev/sumo/) (Simulation of Urban MObility) simulates each vehicle on each lane, second by
second. Its network file is `.net.xml`. SUMO's program `netconvert` makes it from three **plain XML** files: the
nodes (`.nod.xml`), the edges (`.edg.xml`) and the connections between lanes (`.con.xml`).

`to_sumo` writes the three plain files, then runs `netconvert` on them. The extra `sim` installs `netconvert` (the
Python package `eclipse-sumo`).

| Argument | Default | Does |
|---|---|---|
| `net_name` | `"network"` | the start of each file name |
| `run_netconvert` | `True` | `False`: only the plain XML, with no SUMO program |
| `connections` | `True` | writes the connections of the lanes at each junction (from the turn graph and the turn lanes) |
| `config` | none | a netconvert config file of your own |

In [3]:
sumo = net.to_sumo("out/sumo")    # the plain XML files, and the .net.xml of netconvert
sumo                              # the paths and the numbers

SUMO out/sumo: done in 0.7 s

running: netconvert -c out/sumo/network.netccfg


{'nod': PosixPath('out/sumo/network.nod.xml'),
 'edg': PosixPath('out/sumo/network.edg.xml'),
 'n_nodes': 1768,
 'n_edges': 2840,
 'n_connections': 5082,
 'con': PosixPath('out/sumo/network.con.xml'),
 'netccfg': PosixPath('out/sumo/network.netccfg'),
 'net': PosixPath('out/sumo/network.net.xml')}

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-output.svg" width="18" height="18" alt=""> **What the output shows**
>
> The dict has the paths of the three plain files, of the netconvert config (`.netccfg`) and of SUMO's network (`.net.xml`). The plain files have 1,768 nodes, 2,840 edges and 5,082 connections: the same nodes and edges as the network, and one connection for each arc of the turn graph that a car can use (chapter 7).

The check: SUMO's own network file must have each edge of duckOSM. `duckosm.sim.edge_map` reads the `.net.xml`
and gives the `edge_id` of each SUMO edge (netconvert can split an edge into parts, for example `123#1`; the map
gives the `edge_id` of each part).

In [4]:
ids = duckosm.sim.edge_map(sumo)                 # the SUMO id and the edge_id of each SUMO edge
len(ids), ids.edge_id.nunique()                  # the rows, and the different edge ids in them

(2840, 2840)

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-output.svg" width="18" height="18" alt=""> **What the output shows**
>
> SUMO's network has 2,840 edges, and they are 2,840 different edge ids: netconvert kept each duckOSM edge, with its `edge_id` as its SUMO id.

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-warning.svg" width="18" height="18" alt=""> **Warning**
>
> Without the extra `sim` (or a SUMO install), `to_sumo` stops with "netconvert not found". Install `duckosm[sim]`, or use `run_netconvert=False` and run `netconvert` on the plain files later.

## GMNS

[GMNS](https://github.com/zephyr-data-specs/GMNS) (General Modeling Network Specification) is a standard set of tables
for transport planning: `node`, `link`, `lane`, `movement` (a turn at a junction) and more. Planning tools such as
path4gmns, AequilibraE and DTALite read it.

`to_gmns` writes the tables into a DuckDB file, one schema for each mode (`gmns_driving` ...). `to_csv` also writes
each table as a CSV file, the form of the standard.

| Argument | Default | Does |
|---|---|---|
| `to_csv` | none | a folder: the CSV files too |
| `modes` | all | the modes, when you call it on the network (`mode` gives one) |
| `lane_geometry` | `True` | a line for each lane |
| `combined` | `False` | also one network of all modes together |

In [5]:
gmns = net.to_gmns("out/gmns.duckdb", to_csv="out/gmns")         # the GMNS tables in a DuckDB file and as CSV
gmns["modes"]                                                     # the rows of each table

GMNS out/gmns.duckdb: done in 38.6 s

{'driving': {'node': 1772,
  'link': 2858,
  'lane': 3318,
  'movement': 3755,
  'signal_controller': 1,
  'curb_seg': 0}}

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-output.svg" width="18" height="18" alt=""> **What the output shows**
>
> The GMNS network of driving has 1,772 nodes, 2,858 links, 3,318 lanes and 3,755 movements. It has 18 links more than `driving.edges`: for example the other direction of a one-way road that a bus route uses. The export also makes the lanes and the movements. It took more than 25 s here. Each other export of the chapter takes less than 1 s.

The check: `duckosm.planning.check_gmns` runs the rules of the standard on the tables. Examples: each link starts at
its `from_node`, each lane number is correct, each movement turns where its inbound link ends. It gives one row for
each rule: the schema, the rule, the number of bad rows and the number of rows.

In [6]:
checks = duckosm.planning.check_gmns("out/gmns.duckdb")             # (schema, check, bad rows, rows) for each check
len(checks), [c for c in checks if c[2] > 0]                        # the number of checks, and the checks with bad rows

(17, [])

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-output.svg" width="18" height="18" alt=""> **What the output shows**
>
> `check_gmns` ran 17 checks, and no check found a bad row (the empty list).

## MATSim

[MATSim](https://www.matsim.org) simulates the trips of each person of a population for a whole day. Its network is
one XML file (`network.xml`, gzip by default) with `<node>` and `<link>` elements. A link has its length, its free
speed (m/s), its capacity and its lanes.

| Argument | Default | Does |
|---|---|---|
| `crs` | the UTM zone of the area | the coordinate system of the nodes (MATSim needs metres) |
| `gzip` | `True` | writes `.xml.gz` |

In [7]:
import gzip
import xml.etree.ElementTree as ET                                   # to read the XML back

matsim = net.to_matsim("out/matsim_network.xml.gz")                  # the MATSim network
matsim, len(ET.parse(gzip.open("out/matsim_network.xml.gz")).findall(".//link"))    # its numbers, and the links in the file

MATSim out/matsim_network.xml.gz: done in 0.1 s

({'nodes': 1768, 'links': 2840, 'crs': 'EPSG:32632'}, 2840)

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-output.svg" width="18" height="18" alt=""> **What the output shows**
>
> The MATSim network has 1,768 nodes and 2,840 links, in UTM zone 32N (`EPSG:32632`, in metres). The XML file has 2,840 `<link>` elements: one link for each edge.

## OpenDRIVE (experimental)

[ASAM OpenDRIVE](https://www.asam.net/standards/detail/opendrive/) (`.xodr`) describes roads for driving simulators:
the reference line of each road, its lanes and their widths. Simulators such as CARLA and esmini read it. The export
is experimental: it writes one road for each edge, and the junctions only with `junctions=True`.

| Argument | Default | Does |
|---|---|---|
| `crs` | the UTM zone of the area | the coordinate system (metres) |
| `junctions` | `False` | `True`: also the OpenDRIVE junctions |

In [8]:
odr = net.to_opendrive("out/monaco.xodr")                    # the OpenDRIVE file
odr, len(ET.parse("out/monaco.xodr").findall("road"))       # its numbers, and the roads in the file

OpenDRIVE out/monaco.xodr: done in 0.1 s

({'roads': 2840, 'crs': 'EPSG:32632'}, 2840)

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-output.svg" width="18" height="18" alt=""> **What the output shows**
>
> The OpenDRIVE file has 2,840 roads, in UTM zone 32N. The XML has 2,840 `<road>` elements: one road for each edge.

## GeoPackage, GeoParquet and shapefiles

A GIS (QGIS, ArcGIS) and GeoPandas read standard GIS files. `to_gis` writes the edges and the nodes of each mode as
layers (`edges_driving`, `nodes_driving` ...) and the boundary of the area.

| Argument | Default | Does |
|---|---|---|
| `fmt` | `"gpkg"` | `"gpkg"`: one GeoPackage file; `"geoparquet"`: a folder of GeoParquet files; `"shp"`: a folder of shapefiles |
| `modes` | all | the modes (`mode` gives one) |
| `boundary` | `True` | also the layer `boundary` |
| `srs` | `"EPSG:4326"` | the coordinate system of the file |

In [9]:
import duckdb                                                # to read the layers back

gis = net.to_gis("out/monaco.gpkg")                          # the GeoPackage
con = duckdb.connect()                                       # DuckDB with the extension spatial
con.load_extension("spatial")
{layer: con.execute(f"SELECT count(*) FROM st_read('out/monaco.gpkg', layer='{layer}')").fetchone()[0]
 for layer in gis["layers"]}                                 # the rows of each layer, as DuckDB reads them

GIS out/monaco.gpkg: done in 0.2 s

{'edges_driving': 2840, 'nodes_driving': 1768, 'boundary': 1}

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-output.svg" width="18" height="18" alt=""> **What the output shows**
>
> The GeoPackage has three layers: 2,840 edges, 1,768 nodes and the boundary (1 row). DuckDB reads each layer with the same number of rows as the network.

GeoParquet is a Parquet file with a geometry column. It is fast to read for DuckDB, GeoPandas and the cloud data
tools. `fmt="geoparquet"` writes one file for each layer into a folder.

In [10]:
pq = net.to_gis("out/geoparquet", fmt="geoparquet")         # a folder of GeoParquet files
len(pq["files"]), pq["layers"]["edges_driving"], con.sql("SELECT count(*) FROM 'out/geoparquet/edges_driving.parquet'").fetchone()[0]

GIS out/geoparquet: done in 0.1 s

(15, 2840, 2840)

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-output.svg" width="18" height="18" alt=""> **What the output shows**
>
> The folder has 15 files. The edges file has 2,840 rows, also when DuckDB reads it. The folder also has the layers of the base map (`features_buildings`, `features_streets` ...), which the GeoPackage of the cell before does not have.

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-tip.svg" width="18" height="18" alt=""> **Tip**
>
> A shapefile cuts each column name to 10 characters, and each layer is a set of files. Use GeoPackage or GeoParquet when the tool can read them.

## QuackOSM table and PMTiles

The calls above export the **network**. `duckosm.gis.quack` exports the **OSM data**: it reads the OSM file, not the
network, and gives all OSM features (nodes, ways, relations) with all their tags, in the format of
[QuackOSM](https://github.com/kraina-ai/quackosm). It takes the `Area` of `duckosm.get` (or the path of an OSM file).

| Argument | Default | Does |
|---|---|---|
| `out`, `name` | `"."`, `"area"` | the folder and the start of the file name (`<name>_quack.parquet`) |
| `formats` | GeoParquet | the formats of the table: `"geoparquet"`, `"gpkg"` ... |
| `gdal` | `False` | `True`: the five layers of GDAL's OSM driver in place of the table |
| `fixes` | none | the fixes file (chapter 5) |

In [11]:
area = duckosm.get(osm_id=2220322, name="monaco", out="data")                                          # the OSM file (from the cache)
quack = duckosm.gis.quack(area, out="out", name="monaco", formats=["geoparquet"])    # the QuackOSM table
con.sql(f"SELECT split_part(feature_id, '/', 1) AS type, count(*) AS features FROM '{quack[0]}' GROUP BY 1 ORDER BY 1").df()    # the features by OSM type

cut: 34461 nodes, 5685 ways, 339 relations

out/monaco_quack.parquet

,type,features
0,node,3994
1,relation,59
2,way,5606


> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-output.svg" width="18" height="18" alt=""> **What the output shows**
>
> The QuackOSM table of Monaco has 3,994 nodes (points with tags: shops, stops, trees ...), 5,606 ways and 59 relations. This is all of the OSM data with tags, not only the roads: a building, a park and a bench are features too.

**PMTiles** is one file of vector tiles: a base map that a web map (MapLibre, OpenLayers) reads directly from a web
server, with no tile server. The command `duckosm quack --pmtiles` makes it. The SDK has no Python call for it yet,
so the cell runs the command.

In [12]:
import subprocess, sys
from pathlib import Path

subprocess.run([sys.executable, "-m", "duckosm", "quack", str(area.pbf), "--pmtiles", "-o", "out", "--name", "monaco"],
               check=True)                                             # the command duckosm quack --pmtiles
Path("out/monaco.pmtiles").stat().st_size, Path("out/monaco.pmtiles").read_bytes()[:7]    # the size; a PMTiles file starts with "PMTiles"

     0.0 s  quack --pmtiles: the boundaries
     0.0 s  quack 1: the ways and the relations
     0.1 s  quack 2: the nodes and the points
     0.1 s  quack 3: the ways
     0.1 s  quack 4a: the lines of the relations
     0.1 s  quack 4b: the relations
     0.1 s  quack --pmtiles: the rows of the tiles
     0.1 s  quack --pmtiles: the tiles
quack --pmtiles: 9659 features, 59 tiles, 0.2 MB in 0.10 s
  out/monaco.pmtiles


(219713, b'PMTiles')

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-output.svg" width="18" height="18" alt=""> **What the output shows**
>
> The command made 59 tiles of 9,659 features in 0.1 s. The file is 0.2 MB (219,713 bytes), and its first seven bytes are `PMTiles`.

## The files of the chapter

Each export wrote its files into the folder `out`. The last cell lists them with their sizes.

In [13]:
sorted((str(p), round(p.stat().st_size / 1e6, 2)) for p in Path("out").rglob("*") if p.is_file() and p.suffix != ".csv")    # path and MB

[('out/geoparquet/boundary.parquet', 0.01),
 ('out/geoparquet/edges_driving.parquet', 0.34),
 ('out/geoparquet/features_buildings.parquet', 0.18),
 ('out/geoparquet/features_land.parquet', 0.07),
 ('out/geoparquet/features_ocean.parquet', 0.01),
 ('out/geoparquet/features_paved.parquet', 0.02),
 ('out/geoparquet/features_place_labels.parquet', 0.0),
 ('out/geoparquet/features_pois.parquet', 0.05),
 ('out/geoparquet/features_public_transport.parquet', 0.0),
 ('out/geoparquet/features_sites.parquet', 0.01),
 ('out/geoparquet/features_streets.parquet', 0.3),
 ('out/geoparquet/features_traffic.parquet', 0.02),
 ('out/geoparquet/features_water_lines.parquet', 0.0),
 ('out/geoparquet/features_water_polygons.parquet', 0.02),
 ('out/geoparquet/layers.json', 0.0),
 ('out/geoparquet/nodes_driving.parquet', 0.05),
 ('out/gmns.duckdb', 9.45),
 ('out/matsim_network.xml.gz', 0.1),
 ('out/monaco.gpkg', 1.28),
 ('out/monaco.pmtiles', 0.22),
 ('out/monaco.xodr', 2.1),
 ('out/monaco_quack.parquet', 0.72

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-output.svg" width="18" height="18" alt=""> **What the output shows**
>
> The largest file is the GMNS database (9.2 MB): it has the lanes, the movements and a line for each lane. SUMO's `.net.xml` is 5.2 MB. The MATSim network is the smallest export of the network (0.1 MB), because it is compressed with gzip.

## Try it yourself

**1.** Export the walking network of Monaco to MATSim. Build it first with `modes=["walking"]`.

<details><summary>Show the answer</summary>

```python
walk = duckosm.network(osm_id=2220322, modes=["walking"], name="monaco_walk")
walk.walking.to_matsim("out/matsim_walking.xml.gz")
```
</details>

**2.** Write the driving network as GeoParquet files in the coordinate system of UTM zone 32N (`EPSG:32632`), and read the edges with GeoPandas.

<details><summary>Show the answer</summary>

```python
import geopandas as gpd
net.to_gis("out/utm", fmt="geoparquet", srs="EPSG:32632")
gpd.read_parquet("out/utm/edges_driving.parquet").crs
```
</details>

**3.** Write only the plain XML of SUMO, with no netconvert, into the folder `out/plain`, and count the `<edge>` elements of the `.edg.xml`.

<details><summary>Show the answer</summary>

```python
plain = net.to_sumo("out/plain", run_netconvert=False)
len(ET.parse(plain["edg"]).findall("edge"))
```
</details>

## Words in this chapter

| Word | Meaning |
|---|---|
| export | a call that writes the network in the format of another tool |
| plain XML | the three input files of SUMO's `netconvert`: nodes, edges, connections |
| `netconvert` | the program of SUMO that makes the `.net.xml` network |
| link | the word of MATSim and GMNS for an edge |
| movement | a turn from one link to another at a junction, in GMNS |
| layer | one table of a GIS file, for example `edges_driving` |
| GeoParquet | a Parquet file with a geometry column |
| vector tiles | a map cut into small squares of geometry, for web maps; PMTiles holds them in one file |

## Summary

| Call | Does |
|---|---|
| `net.to_sumo(out_dir, run_netconvert=True)` | the SUMO plain XML and the `.net.xml` |
| `duckosm.sim.edge_map(files)` | the `edge_id` of each SUMO edge or MATSim link |
| `net.to_gmns(out_path, to_csv=None)` | the GMNS tables in DuckDB, and as CSV |
| `duckosm.planning.check_gmns(path)` | the checks of the GMNS tables |
| `net.to_matsim(out_path)` | the MATSim network |
| `net.to_opendrive(out_path)` | the OpenDRIVE file (experimental) |
| `net.to_gis(out, fmt="gpkg")` | a GeoPackage, GeoParquet files or shapefiles |
| `duckosm.gis.quack(area, formats=...)` | the QuackOSM table of the OSM file |
| `duckosm quack --pmtiles` | a PMTiles file of vector tiles (the command) |

The reference: [duckosm.sim](../reference/sdk/sim.md), [duckosm.planning](../reference/sdk/planning.md), [duckosm.hdmap](../reference/sdk/hdmap.md), [duckosm.gis](../reference/sdk/gis.md). **Next:** [7. Routing](07_routing.md), the fastest route between two roads, two points or two names.